# Take-Home Case: The Mortgage Book of Aare-Säntis Regionalbank AG

**EAIF: AI for Finance. Team take-home between the classes on linear/logistic regression and on advanced supervised learning**

You are the newly formed data team of Aare-Säntis Regionalbank AG, a fictional regional bank in the Swiss Mittelland. The bank has 10,000 residential mortgages on its books, originated between 2019 and 2022, spread over seven cantons. Until now, the bank's credit decisions have rested on a vendor valuation model, a handful of ratio rules and the judgement of the branch advisers. This morning the Chief Risk Officer sent the team its first assignment.

> **Memo from the CRO**
>
> To the data team. Welcome aboard. Three things, in order of urgency.
>
> 1. Our collateral valuations come from a vendor model we cannot inspect. Build one we can.
> 2. About seven in a hundred of our recent mortgages ran into payment trouble within three years. Tell me which applications carry that risk, and where we should set the approval bar, in francs.
> 3. The risk committee has read that banks use "machine learning" for this. Show me whether the more flexible methods actually do better on our book, and whether we could defend deploying one.
>
> I do not need a slide deck. I need numbers I can trust and one paragraph per question that I can read out to the committee.
>
> Head of Risk

This notebook is your working file for the assignment. Part A answers the first two requests with the methods you already know, linear and logistic regression. Part B answers the third with the methods introduced in the next class. Every section ends with what you, the analyst, tell the CRO.

## The data

Two files are loaded from the course repository. `mortgages.csv` is the bank's book: 10,000 mortgages originated 2019 to 2022, each with its outcome after 36 months. `applications_2026.csv` holds this year's 500 applications. It has the same columns as the book except two: the outcome, which does not exist yet because the bank has not decided on them, and one further column that Part A2 comes to. The applications file has 22 columns where the book has 24. The full column dictionary is in [`data/mortgage2026/README.md`](https://github.com/umatter/EDFB/blob/main/data/mortgage2026/README.md); the groups of columns are these.

| Group | Columns |
|---|---|
| Property | `canton`, `property_type`, `living_area_m2`, `rooms`, `year_built`, `distance_center_km`, `energy_label`, `purchase_price` |
| Borrower | `household_income`, `age`, `employment`, `years_client` |
| Loan | `loan_amount`, `rate_type`, `fixed_years`, `interest_rate`, `amortisation`, `origination_year` |
| Derived ratios | `ltv`, `affordability`, `actual_burden` |
| Outcome | `trouble_36m` (1 if the mortgage was 90 days or more in arrears, or was restructured, within 36 months) |

Three ratios do most of the work in Swiss mortgage lending, and all three are in the file.

The loan-to-value ratio compares the loan with the price of the property: `ltv = loan_amount / purchase_price`. Swiss banks normally finance at most 80 % of the price, and the part of the loan above two-thirds of the price must be amortised within 15 years.

The affordability ratio is the Swiss lending rule for whether the household can carry the loan through a rise in interest rates. It does not use the interest rate actually agreed but an imputed rate of 5 %, adds 1 % of the purchase price per year for maintenance, adds the amortisation of the part above two-thirds LTV spread over 15 years, and divides the sum by gross household income:

`affordability = (0.05 × loan_amount + 0.01 × purchase_price + amortisation per year) / household_income`

The rule says the ratio must not exceed one third. A household with CHF 150,000 gross income and a CHF 800,000 loan on a CHF 1,000,000 property carries 40,000 of imputed interest, 10,000 of maintenance and about 8,900 of amortisation per year, which is 0.39 of its income, above the bar.

The actual burden is what the household pays in interest today, at the agreed rate: `actual_burden = interest_rate × loan_amount / household_income`. With rates mostly between 1 and 2 %, it is a fraction of the affordability ratio, which is exactly why the imputed rate exists.

> **Simulated data.** The two files were generated by the course for this case. There is no real bank, property or household behind any row. The rate of payment trouble in the book is several times higher than in a real Swiss mortgage book so that the models have enough cases to learn from; the ratios, prices and incomes are in a realistic range but were not taken from any real data source.

One more thing to know before you start: the book contains one column that the bank does *not* know at the moment it decides on an application. Part A2 deals with it.

## How to work through this notebook

Plan three hours for a team of three to four. Make a copy of the notebook in your Drive (File, Save a copy in Drive) and work in the copy. Run the given cells one after the other and read them, including the comments; they are the worked part of the case and they are where the methods are explained. The exercises are marked `### Exercise n` and each one is followed by a code cell that holds only a comment. Fill that cell and leave the given cells as they are, because later parts of the notebook use the objects they create.

Rotate who types for each part, so that nobody sits through the whole case as a spectator. In the next class, any member of the team can be asked to explain any cell, worked or exercise, so make sure everyone can. The notebook is discussed in that class; it is not graded.

Every choice of columns in this case follows one rule, which you will meet again in every part:

**At the moment the bank decides on an application, which columns are already known?**

Everything a model uses must pass that test. A model that is fed a column the bank only learns afterwards looks excellent in the notebook and is useless at the counter.

## Roadmap

| Part | Question | Method | Exercises |
|---|---|---|---|
| A1 | Is our collateral valued right? | Linear regression | 1, 2 |
| A2 | Which applications will run into payment trouble, and where is the approval bar in francs? | Logistic regression | 3, 4 |
| B | Do the flexible methods do better on our book? | LASSO, decision tree, random forest, gradient boosting and XGBoost, SVM, comparison | |
| C | Your turn on the new methods | The methods of Part B, and reflection questions for the class | 5 to 8 |

## Setup

In [ ]:
# Setup: Colab's preinstalled stack only, nothing to install
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance, DecisionBoundaryDisplay
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, roc_auc_score,
                             roc_curve, confusion_matrix, precision_score, recall_score)
import xgboost as xgb

np.random.seed(0)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
sns.set_theme(style="whitegrid")

# The two error costs the CRO gave us, in CHF, used throughout the notebook
COST_FN = 60_000   # a loan that runs into trouble was approved: expected loss
COST_FP = 8_000    # a loan that would have been fine was rejected: lost margin

print("Setup complete. scikit-learn", __import__("sklearn").__version__, "| xgboost", xgb.__version__)

In [ ]:
# Load the two data files from the course repository
DATA = "https://raw.githubusercontent.com/umatter/EDFB/main/data/mortgage2026/"
book = pd.read_csv(DATA + "mortgages.csv")
apps = pd.read_csv(DATA + "applications_2026.csv")
print("book:", book.shape, "| applications:", apps.shape)
book.head()

In [ ]:
# Types, missing values, and the outcome's base rate
print(book.dtypes, "\n")
print("missing values:", int(book.isna().sum().sum()))
print(f"trouble rate in the book: {book.trouble_36m.mean():.3%}  ({book.trouble_36m.sum()} of {len(book)})")
book.describe().T

# Part A: Recap

## A1. Is our collateral valued right? Linear regression

The bank lends against the property. If the borrower stops paying, the bank sells the property and recovers what it can, so the question behind every mortgage is what the property is worth, as opposed to what the buyer paid for it. Today that question is answered by a vendor model that returns a number and no explanation. The CRO's first request is a valuation the bank can inspect.

The standard tool for this is a hedonic model: the price of a property is written as the sum of the prices of its characteristics. A linear regression is exactly such a model, and it is inspectable by construction. Its coefficients are a price per square metre, a premium or discount per canton, a discount per kilometre from the regional centre, a premium for a house over an apartment. A valuer can read those numbers, argue with them, and compare them with what the market pays.

We follow the steps of the first supervised-learning class.

1. Look at the data, in a plot, before fitting anything.
2. Pick the target Y (`purchase_price`) and the features X (the property columns).
3. Split the book into a training set and a test set.
4. Fit the model on the training set.
5. Test it on the held-out data, and put its RMSE next to the RMSE of a baseline that knows nothing.

The features are all property characteristics, which the bank knows when the application arrives, so the decision-time rule is satisfied.

In [ ]:
# Step 1: look at the relationship we want to model. Price against living area, one point per mortgage.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(book.living_area_m2, book.purchase_price / 1e6, s=6, alpha=0.3)
ax.set_xlabel("living area (m²)")
ax.set_ylabel("purchase price (CHF million)")
ax.set_title("The bank's book: purchase price against living area")
plt.show()

In [ ]:
# Step 2: the simplest model. One X, one slope: CHF per square metre, averaged over everything else.
uni = LinearRegression().fit(book[["living_area_m2"]], book.purchase_price)
print(f"price = {uni.intercept_:,.0f} + {uni.coef_[0]:,.0f} x living area")
print(f"R² on the full book: {uni.score(book[['living_area_m2']], book.purchase_price):.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(book.living_area_m2, book.purchase_price / 1e6, s=6, alpha=0.3)
grid = np.linspace(45, 320, 50).reshape(-1, 1)
ax.plot(grid, uni.predict(pd.DataFrame(grid, columns=["living_area_m2"])) / 1e6, color="C3", lw=2)
ax.set_xlabel("living area (m²)"); ax.set_ylabel("purchase price (CHF million)")
ax.set_title("Simple linear regression: one slope for all cantons")
plt.show()

The slope is the whole model. In this run it says that one additional square metre of living area adds CHF 7,589 to the price, averaged over every canton, every building age and every location in the book, and living area alone explains 48 % of the variation in prices (an R² of 0.484). The line runs through the middle of the cloud, but the cloud is wide: at 150 m² the book holds properties that sold for well under one million and others that sold for close to two.

One slope is not enough because a square metre does not cost the same everywhere. A square metre in the canton of Zurich and one in the canton of Solothurn are priced in different markets. A house built in 1960 and one built in 2018 differ in what a buyer pays, as do a flat next to the station and one twenty kilometres out. The simple model averages over all of that, and the spread around the line is the price of averaging.

The multivariate model puts these characteristics in as further columns of X. The numeric ones (`living_area_m2`, `year_built`, `distance_center_km`) enter as they are. The categorical ones (`canton`, `property_type`, `energy_label`) become 0/1 dummy columns, one per level, with one reference level dropped per column, exactly as in the logistic-regression class: the coefficient of each dummy is then the premium relative to the dropped level. `pd.get_dummies(..., drop_first=True)` drops the alphabetically first level, so the references are canton AG, property type `apartment` and energy label A.

In [ ]:
# Step 3: the multivariate model. Categorical columns become 0/1 dummies (one reference level dropped each).
price_features = ["canton", "property_type", "living_area_m2", "year_built", "distance_center_km", "energy_label"]
Xp = pd.get_dummies(book[price_features], drop_first=True).astype(float)
yp = book.purchase_price

Xp_train, Xp_test, yp_train, yp_test = train_test_split(Xp, yp, test_size=0.3, random_state=42, stratify=book.canton)
lin = LinearRegression().fit(Xp_train, yp_train)

coef = pd.Series(lin.coef_, index=Xp.columns).sort_values()
print("intercept:", f"{lin.intercept_:,.0f}")
coef.round(0).to_frame("CHF per unit")

Every row of the table is a price the bank can read. In this run, one square metre of living area adds CHF 6,705, now holding canton, building age, location and energy label fixed, which is about CHF 900 less than the slope of the simple model. The canton dummies are premiums relative to Aargau, the dropped reference: a property in the canton of Zurich sells for about CHF 403,000 more than the same property in Aargau, one in Solothurn for about CHF 187,000 less. Each kilometre further from the regional centre takes about CHF 17,900 off the price. A house sells for about CHF 96,000 more than an apartment with the same area, age, canton and location, and each energy label below A carries its own discount.

These numbers are the inspectable model the CRO asked for. A valuer who disagrees with the Zurich premium or the discount per kilometre can say so, in francs, and the bank can check the number against recent transactions.

In [ ]:
# Step 4: test the model on data it has not seen, next to the baseline "predict the training mean"
def rmse_of(y, pred):
    return float(np.sqrt(mean_squared_error(y, pred)))

pred_test = lin.predict(Xp_test)
baseline = np.full(len(yp_test), yp_train.mean())
print(f"RMSE  linear model : CHF {rmse_of(yp_test, pred_test):>10,.0f}   (R² {r2_score(yp_test, pred_test):.3f})")
print(f"RMSE  predict mean : CHF {rmse_of(yp_test, baseline):>10,.0f}   (R² {r2_score(yp_test, baseline):.3f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(yp_test / 1e6, pred_test / 1e6, s=6, alpha=0.3)
lim = [0, yp_test.max() / 1e6]
axes[0].plot(lim, lim, color="C3", lw=1)
axes[0].set_xlabel("actual price (CHF million)"); axes[0].set_ylabel("predicted price (CHF million)")
axes[0].set_title("Test set: predicted against actual")
axes[1].scatter(pred_test / 1e6, (yp_test - pred_test) / 1e6, s=6, alpha=0.3)
axes[1].axhline(0, color="C3", lw=1)
axes[1].set_xlabel("predicted price (CHF million)"); axes[1].set_ylabel("residual (CHF million)")
axes[1].set_title("Residuals fan out as the price grows")
plt.show()

The test set holds 3,000 mortgages the model has never seen. In this run the linear model's RMSE on the test set is CHF 154,786, against CHF 426,031 for the baseline that predicts the training mean for every property, and it explains 87 % (an R² of 0.868) of the variation in test prices where the baseline explains none. For a property that the model values at CHF 900,000, an error of one RMSE means that the price such a property fetches is plausibly anywhere between about CHF 745,000 and CHF 1,055,000, which is the uncertainty the bank has to keep in mind when it sets the loan-to-value ratio.

The right-hand plot shows something the RMSE hides. The residuals are not a band of constant width; they fan out. Cheap properties are missed by a small amount, expensive ones by a large amount, and the error grows in proportion to the price. That is the signature of a multiplicative relationship: a Zurich premium is more naturally a percentage of the price than a fixed sum in francs, and so is the discount for an old building. A model in levels cannot express that, which is why it is imprecise exactly where the bank's exposures are largest. Exercise 1 takes the hint.

What the analyst tells the CRO: a linear regression on six property characteristics values the collateral to within about CHF 155,000 on unseen properties, every coefficient is a price in francs that a valuer can inspect and challenge, and the model's main weakness, its imprecision for expensive properties, can be addressed by modelling the price in logarithms, which is the next step.

### Exercise 1: A model in logarithms

Fit the same multivariate model on `np.log(yp_train)` instead of `yp_train`. Predict on the test set, transform the predictions back with `np.exp`, and report the RMSE in CHF next to the RMSE of the model in levels. Then plot the residuals of the log model against its predictions as in the cell above.

*Deliverable:* the two RMSE values, and one sentence on which model the bank should use and why (look at the residual plot, not only at the RMSE).

In [ ]:
# Exercise 1: your code here
# lin_log = ...

### Exercise 2: Did the buyer overpay? A feature for Part B

Use your log model to predict a value for **every** property in the book (`Xp`, not only the test set) and compute `overpayment = purchase_price / predicted value`. A value above 1 means the buyer paid more than comparable properties cost. Store it as `book["overpayment"]`. Then compare the trouble rate of the mortgages in the top 10 % of `overpayment` with the rest.

*Deliverable:* the two trouble rates and one sentence on whether the valuation model tells the bank something about repayment risk. (Part B recomputes this column itself, so the notebook keeps running if you skip this.)

In [ ]:
# Exercise 2: your code here
# book["overpayment"] = ...

## A2. Which borrowers run into payment trouble? Logistic regression

The CRO's second request is different in kind from the first. The target is no longer a price but a yes or no: did the mortgage run into payment trouble within 36 months (`trouble_36m` equal to 1) or not. This is classification, and the tool from the second supervised-learning class is logistic regression. It does not predict the outcome itself; it predicts a probability of trouble for each application, and the bank turns that probability into a decision by choosing a threshold above which it declines. The threshold is where the CRO's phrase "in francs" enters, and it is the subject of the second half of this part.

We follow the sequence of that class.

1. Look at the base rate, and at what a model that always predicts the majority class would score.
2. Pick the features, applying the decision-time rule column by column.
3. Split the book into a training and a test set, stratified so that both hold the same share of troubled loans, and standardise the features on the training set only.
4. Fit the logistic regression and read its coefficients as odds ratios.
5. Turn the test-set probabilities into decisions at a threshold and read the confusion matrix.
6. Judge the ranking independently of any threshold with the ROC curve and its AUC, computed from the probabilities.
7. Put a price on the two kinds of error, in CHF, and ask where the threshold should sit.

The decision-time rule bites here for the first time. The column dictionary marks one column of the book as known only afterwards: `reminders_sent`, the number of payment reminders the bank sent during the 36 months. It is the column that the applications file does not have, and the cell below shows why it cannot be a predictor: it is recorded after the outcome it would predict, and it is to a large extent the same event seen from the bank's side. Three further columns are known at decision time but are left out for the reasons the comment gives. `purchase_price` and `loan_amount` already enter through the three ratios, and `origination_year` takes a value in the applications, 2026, that the book never contains, so a coefficient estimated on 2019 to 2022 could not be applied to it. A fourth, `overpayment`, exists only if your team completed Exercise 2; the cell leaves it out so that every team fits the same model here, and Part B brings it in.

In [ ]:
# Which columns does the bank know when it decides? Everything in the file except the outcome and one more.
# reminders_sent counts the payment reminders sent DURING the 36 months: it is a consequence of trouble,
# not a predictor. A model that uses it looks brilliant on the book and is useless for an application.
print(book.groupby("trouble_36m").reminders_sent.mean().rename("mean reminders_sent"))
print()
# purchase_price and loan_amount are known, but they enter through ltv / affordability / actual_burden;
# origination_year is known, but this year's applications come from 2026, a year the book never saw.
# overpayment, if Exercise 2 created it, is left for Part B, so that every team sees the same numbers here.
target = "trouble_36m"
not_features = ["id", "purchase_price", "loan_amount", "origination_year", "reminders_sent", "overpayment", target]
features = [c for c in book.columns if c not in not_features]
print("features used:", features)

In [ ]:
# Base rate and the majority-class baseline: a "model" that approves everyone is right 93% of the time
print(f"trouble rate: {book[target].mean():.3%}")
print(f"accuracy of 'nobody gets into trouble': {1 - book[target].mean():.3%}")

In [ ]:
# Split first (stratified on the outcome), then standardise on the training set only
X = pd.get_dummies(book[features], drop_first=True).astype(float)
y = book[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)
print("train:", X_train.shape, "| test:", X_test.shape, "| columns:", list(X.columns))

In [ ]:
# Logistic regression. Coefficients are log-odds per one standard deviation of the feature (we standardised),
# and exp(coef) is the odds ratio: how much the odds of trouble multiply when the feature rises by one SD.
logit = LogisticRegression(max_iter=2000).fit(X_train_s, y_train)
p_logit = logit.predict_proba(X_test_s)[:, 1]

odds = pd.DataFrame({"coef (log-odds per SD)": logit.coef_[0], "odds ratio": np.exp(logit.coef_[0])},
                    index=X.columns).sort_values("coef (log-odds per SD)", key=abs, ascending=False)
odds.round(3)

Every row of the table is an odds ratio, and an odds ratio is a multiplier on the odds of trouble when the feature rises by one standard deviation, holding the other columns fixed. In this run the largest is `affordability`: one standard deviation more, which is about 0.13 more of gross income needed to carry the imputed cost, multiplies the odds of trouble by 2.7. The Swiss rule looks at the right ratio. Next come `household_income` with 1.7 and `rate_type_saron` with 1.7, then `ltv` with 1.6, where one standard deviation is about ten points of loan-to-value.

Two of these need care. The income row points the way a reader may not expect: more income, higher odds of trouble. It is read holding the three ratios fixed, and at the same affordability and loan-to-value more income means a larger loan, so the row says that larger exposures at the same ratios go wrong more often, not that the bank should prefer poorer households. A coefficient on a column that also sits in the denominator of three other features is the clearest warning in the table against reading any row on its own. The SARON row is a 0/1 column, so one standard deviation is not a switch from a fixed to a variable rate; that switch is about two standard deviations, and the odds of trouble for a SARON mortgage are about 2.9 times those of a comparable fixed-rate one.

An odds ratio below 1 is protective. The Zurich dummy has an odds ratio of 0.7 per standard deviation, and as with SARON the full switch from Aargau to Zurich is a larger effect in the same direction: a comparable property in Zurich carries clearly lower odds of trouble. One standard deviation more living area, about 38 m², multiplies the odds by 0.72. At the bottom of the table `rooms` (0.97), `years_client` (0.97) and the energy-label dummies (between 0.88 and 1.08, in no order from B to G) sit close to 1. A column with an odds ratio of 1 changes nothing, and this is the first hint that not every column in the file carries information about repayment. Part B returns to it with a method that selects features.

In [ ]:
# From probabilities to decisions: the 0.5 threshold, the confusion matrix and the two error rates
pred_05 = (p_logit >= 0.5).astype(int)
cm = confusion_matrix(y_test, pred_05)
print("confusion matrix (rows: actual 0/1, columns: predicted 0/1)\n", cm)
print(f"accuracy  : {accuracy_score(y_test, pred_05):.3%}   (majority baseline {1 - y_test.mean():.3%})")
print(f"recall    : {recall_score(y_test, pred_05):.3%}   share of troubled loans the model flags")
print(f"precision : {precision_score(y_test, pred_05, zero_division=0):.3%}   share of flagged loans that are troubled")

In [ ]:
# ROC curve and AUC, computed from the probabilities, never from the 0/1 predictions
fpr, tpr, thr = roc_curve(y_test, p_logit)
auc_logit = roc_auc_score(y_test, p_logit)
print(f"AUC on the test set: {auc_logit:.3f}   (coin flip: 0.500)")
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, lw=2, label=f"logit, AUC = {auc_logit:.3f}")
ax.plot([0, 1], [0, 1], "--", color="grey", label="coin flip, AUC = 0.5")
ax.set_xlabel("false positive rate (good loans flagged)"); ax.set_ylabel("true positive rate (troubled loans flagged)")
ax.set_title("ROC curve on the test set"); ax.legend()
plt.show()

In [ ]:
# The two errors have different prices. Expected cost per 1,000 applications at a given threshold:
def cost_per_1000(y_true, prob, threshold):
    pred = (np.asarray(prob) >= threshold).astype(int)
    fn = int(((pred == 0) & (np.asarray(y_true) == 1)).sum())   # approved, ran into trouble
    fp = int(((pred == 1) & (np.asarray(y_true) == 0)).sum())   # rejected, would have been fine
    return (fn * COST_FN + fp * COST_FP) / len(y_true) * 1000

def best_threshold(y_true, prob, grid=np.arange(0.02, 0.51, 0.01)):
    costs = [cost_per_1000(y_true, prob, t) for t in grid]
    return float(grid[int(np.argmin(costs))])

print(f"cost per 1,000 applications at threshold 0.5   : CHF {cost_per_1000(y_test, p_logit, 0.5):,.0f}")
print(f"cost per 1,000 applications approving everyone : CHF {cost_per_1000(y_test, p_logit, 1.01):,.0f}")

At the conventional threshold of 0.5 the model declines an application only when it thinks trouble is more likely than not, and in this run that is almost nobody: 48 of the 3,000 test applications, of which 28 did run into trouble and 20 would have been fine. The other 180 troubled loans pass. Recall is 13.5 %, precision 58.3 %, and accuracy is 93.3 % against 93.1 % for the baseline that approves everyone, which is the number the CRO must never be shown on its own. The ROC curve tells a different story. The AUC is the probability that a randomly chosen troubled loan receives a higher predicted probability than a randomly chosen good one, so 0.5 is a coin flip and 1 is a perfect ranking. An AUC of 0.816 in this run means that the model's ranking of the applications carries real information, and the 0.5 threshold throws most of it away.

The costs say the same in francs. At 0.5 the expected cost is CHF 3,653,333 per 1,000 applications, against CHF 4,160,000 for approving everyone. Almost all of it, CHF 3.6 million, is the 180 approved loans that went into trouble; the 20 rejected good loans add CHF 53,333. The reason is the CRO's price list. A troubled loan costs CHF 60,000 and a rejected good one CHF 8,000, so a troubled loan costs 7.5 times a lost customer. Declining pays as soon as the expected loss from approving, the probability of trouble times CHF 60,000, exceeds the expected loss from declining, the probability that the loan would have been fine times CHF 8,000. That happens at a probability of trouble of 8,000 / 68,000, about 12 %. The bar belongs well below 0.5, and Exercise 3 finds it empirically by sweeping the threshold.

One deliberate difference from the logistic-regression class: there we undersampled the majority class of the training set as a teaching device, so that the model saw a balanced world, and corrected the intercept afterwards to get deployment probabilities back. Here we do not. The threshold is chosen in CHF, and a cost calculation needs probabilities that mean what they say: a predicted 14 % must be a 14 % chance of trouble, which a fit on the book's own base rate gives directly, without a correction that is only approximate. With 693 troubled loans in the book there is enough to learn from, and the imbalance is handled where it belongs, at the threshold.

What the analyst tells the CRO: a logistic regression on the eighteen columns known at decision time ranks applications with an AUC of 0.816 on unseen loans, the affordability ratio, the loan-to-value ratio and a variable rate are the strongest signals (income enters only as a proxy for exposure at fixed ratios), and at the textbook bar of 0.5 it would decline almost nobody and cost the bank nearly as much as approving everyone. Because a troubled loan costs 7.5 times a lost good customer, the approval bar belongs near a predicted probability of trouble of 12 % rather than 50 %, and Exercise 3 sets it in francs.

### Exercise 3: The approval bar in francs

Sweep the threshold from 0.02 to 0.50 in steps of 0.01 with `cost_per_1000`, plot the expected cost per 1,000 applications against the threshold, and report the cost-minimising threshold together with the recall, precision and cost at that threshold. (This sweep uses the test set; Part B does it properly on cross-validated training predictions.)

*Deliverable:* the threshold, the three numbers, and one sentence for the CRO on what the bar means: at which predicted probability of trouble does the bank stop approving?

In [ ]:
# Exercise 3: your code here
# grid = np.arange(0.02, 0.51, 0.01)

### Exercise 4: The Swiss rule, checked on our own book

The affordability rule says the imputed cost may not exceed one third of gross income, and the standard maximum loan-to-value is 80 %. On the full book, compute the trouble rate for mortgages above and below the 1/3 affordability line, above and below 80 % LTV, and for the four combinations of the two.

*Deliverable:* a 2×2 table of trouble rates (affordability above/below 1/3 by LTV above/below 0.8) and one sentence on what it shows. Keep this table in mind for Part B.

In [ ]:
# Exercise 4: your code here
# over_aff = book.affordability > 1/3

# Part B: The advanced methods

The CRO's third request is the one the risk committee is most curious about and the one the CRO trusts least. The committee has read that banks use "machine learning" for credit decisions. The CRO wants to know two things: whether the more flexible methods actually do better on this book than the logistic regression of Part A2, and whether the bank could defend deploying one, to its supervisor and to a customer whose application it declined. Both halves count. A model that gains a few points of AUC and cannot be explained to anyone is not an improvement for a bank.

Part B fits five methods from the advanced supervised-learning class one after the other, LASSO, a decision tree, a random forest, gradient boosting and a support vector machine, and then puts all of them in one table next to the logistic regression. Every method is trained on the same 7,000 mortgages and judged on the same 3,000 held-out ones, with the same two numbers as in Part A2, the AUC of its ranking and the cost in CHF of its decisions; a comparison on anything else would not be fair. Each method is introduced by what it does differently from the logit, in the words of the class. LASSO keeps the logit but adds a penalty that shrinks the coefficients and sets some to exactly zero, so it selects the columns. A decision tree asks one yes/no question at a time and can therefore draw the corner that Exercise 4 found. A random forest is a panel of trees, each grown on its own sample of the data and of the columns, that votes. Boosting grows trees that correct each other's mistakes, and a support vector machine looks for the widest margin between the two classes. Part C then hands the same tools to your team.

## B1. LASSO: let the data choose the columns

Part A2 ended with a hint. `rooms`, `years_client` and the energy labels had odds ratios close to 1, and the analyst had chosen the columns by hand. Here we do the opposite and give the model more columns than it needs. To the eighteen features of Part A2 we add four: the `overpayment` ratio of Exercise 2 (recomputed in the cell so that nothing here depends on your exercise code), the logarithm of household income, and two derived columns that sound plausible in a credit meeting, income per room and price per square metre. Some of these carry information, some duplicate a column that is already in, and some are noise. The point is that we do not have to know which in advance.

A plain logistic regression with many columns spreads small coefficients over everything, noise included, and every coefficient it estimates costs precision. LASSO changes the objective. It fits the same logistic regression but adds a penalty λ·Σ|β| on the sum of the absolute coefficients, so that fitting the training data well is traded against keeping the coefficients small. Because the penalty uses absolute values, coefficients do not merely shrink as λ grows; they reach exactly zero and stay there, and a column with a zero coefficient has left the model. The model selects. Ridge regression penalises Σβ² instead, which shrinks every coefficient but never sets one to zero; elastic net mixes the two penalties. scikit-learn writes the strength of the penalty as C = 1/λ, so a *small* C means a *strong* penalty and few surviving columns, and a large C means almost no penalty and the plain logit back. The penalty treats every coefficient alike, so the columns must be on the same scale; the LASSO is fitted on standardised columns, as the logit was.

The first cell builds the wide feature set. The second traces the LASSO path: the model is refitted for thirty values of C from a strong penalty to almost none, and the plot shows each coefficient entering as the penalty relaxes, with the columns that end up large labelled. The third cell lets five-fold cross-validation on the training set choose C by AUC, and reads off which columns survived.

In [ ]:
# A wider feature set: the valuation residual from Part A1 (recomputed here so this cell does not depend on Exercise 2)
lin_log_b = LinearRegression().fit(Xp_train, np.log(yp_train))
book["overpayment"] = book.purchase_price / np.exp(lin_log_b.predict(Xp))
book["log_income"] = np.log(book.household_income)
book["income_per_room"] = book.household_income / book.rooms
book["price_per_m2"] = book.purchase_price / book.living_area_m2

wide_features = features + ["overpayment", "log_income", "income_per_room", "price_per_m2"]
Xw = pd.get_dummies(book[wide_features], drop_first=True).astype(float)
Xw_train, Xw_test = Xw.loc[X_train.index], Xw.loc[X_test.index]     # same rows as before
scaler_w = StandardScaler().fit(Xw_train)
Xw_train_s = pd.DataFrame(scaler_w.transform(Xw_train), columns=Xw.columns, index=Xw_train.index)
Xw_test_s = pd.DataFrame(scaler_w.transform(Xw_test), columns=Xw.columns, index=Xw_test.index)
print(Xw.shape[1], "columns:", list(Xw.columns))

In [ ]:
# The LASSO path: refit for a grid of C (= 1/penalty) and watch the coefficients shrink to zero one by one
import warnings   # penalty="l1" is deprecated from scikit-learn 1.8 and removed in 1.10 (use l1_ratio=1 then); silence only that
warnings.filterwarnings("ignore", message="'penalty' was deprecated")
warnings.filterwarnings("ignore", message="Inconsistent values: penalty")
Cs = np.logspace(-3, 1, 30)
path = np.array([LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=2000)
                 .fit(Xw_train_s, y_train).coef_[0] for C in Cs])

fig, ax = plt.subplots(figsize=(9, 5.5))
for j, name in enumerate(Xw.columns):
    ax.plot(Cs, path[:, j], lw=1.5, label=name if np.abs(path[-1, j]) > 0.15 else None)
ax.set_xscale("log"); ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("C = 1 / penalty strength  (left: strong penalty, right: almost none)")
ax.set_ylabel("coefficient (log-odds per SD)")
ax.set_title("LASSO path: which columns survive as the penalty relaxes"); ax.legend(fontsize=8, ncol=2)
plt.show()
n_nonzero = (np.abs(path) > 1e-6).sum(axis=1)
print(pd.DataFrame({"C": Cs.round(4), "non-zero coefficients": n_nonzero}).iloc[::3].to_string(index=False))

In [ ]:
# Let cross-validation choose C, then read the survivors
warnings.filterwarnings("ignore", message="The default value for l1_ratios")            # two more scikit-learn 1.8 -> 1.10
warnings.filterwarnings("ignore", message="The fitted attributes of LogisticRegressionCV")  # transition notices, nothing else
lasso = LogisticRegressionCV(Cs=Cs, penalty="l1", solver="liblinear", cv=5, scoring="roc_auc",
                             max_iter=2000, random_state=0).fit(Xw_train_s, y_train)
p_lasso = lasso.predict_proba(Xw_test_s)[:, 1]
survivors = pd.Series(lasso.coef_[0], index=Xw.columns)
print(f"chosen C: {lasso.C_[0]:.4f}   |   {int((survivors.abs() > 1e-6).sum())} of {len(survivors)} coefficients non-zero")
print("dropped:", list(survivors.index[survivors.abs() <= 1e-6]))
print(f"test AUC  logit (Part A2 features): {auc_logit:.3f}   LASSO (wide features): {roc_auc_score(y_test, p_lasso):.3f}")
cv_auc = lasso.scores_[1].mean(axis=0)                     # mean cross-validated AUC for each C in Cs, the number CV chose on
i_small, i_best = int(np.abs(Cs - 0.045).argmin()), int(np.argmax(cv_auc))
print(f"mean CV AUC  at C = {Cs[i_small]:.3f}: {cv_auc[i_small]:.4f}   at the chosen C = {Cs[i_best]:.3f}: {cv_auc[i_best]:.4f}")
results = {"logit": (logit, X_train_s, X_test_s), "LASSO logit": (lasso, Xw_train_s, Xw_test_s)}
survivors[survivors.abs() > 1e-6].sort_values(key=abs, ascending=False).round(3).to_frame("coef")

In this run the path table is a story read from left to right. Under a strong penalty, C = 0.001, every coefficient is zero and the model predicts the same number for everyone. The first column to enter, at C = 0.003, is `affordability`; `ltv`, `rate_type_saron` and the two employment dummies follow, and by C = 0.045 thirteen columns are in. Only at the far right, at C = 5.3, has every one of the 33 columns a non-zero coefficient, which is the plain logit. Cross-validation chose C = 0.161. There 26 of the 33 columns survive and seven are dropped: `rooms`, `living_area_m2`, `canton_BE`, `property_type_house`, and three of the four columns we added, `log_income`, `income_per_room` and `price_per_m2`. The three dropped additions duplicate information that is already in the model, log income above all, which is the income column on another scale. The fourth addition, `overpayment`, survived with the fifth largest coefficient, 0.300 per standard deviation, and it explains two changes at the top of the table. In Part A2 living area was protective and income raised the odds at fixed ratios; here living area is gone and the income coefficient has fallen to 0.033. A large loan on a small property relative to income was the logit's way of saying that the buyer paid more than the property is worth, and now the column that says so directly is in the model. A team that removes `overpayment` from `wide_features` and reruns the two cells will see the income coefficient come back.

The selection is not sharp, and that is the second lesson of the table. `years_client` survived with a coefficient of -0.016, and five of the six energy-label dummies with coefficients between -0.017 and 0.021 (only label G, at -0.098, is larger), because at C = 0.161 the penalty is mild. The path table shows thirteen columns at C = 0.045, and the cross-validated AUC there, 0.8212, is within a hair of the 0.8213 at the chosen C: cross-validation found almost nothing to choose between thirteen columns and twenty-six. LASSO tells the analyst which columns can go without changing the ranking; it does not make the model more flexible. The test AUC says the same: 0.823 against the logit's 0.816, less than one point for four new columns and a penalty, and the corner from Exercise 4 is as far beyond a penalised logit as it was beyond the plain one.

What the analyst tells the CRO: given 33 columns, the LASSO kept 26 and discarded the ones that duplicate others, and its ranking of unseen applications is as good as the logistic regression's (an AUC of 0.823 against 0.816 on the same loans), with the same signals near the top, loan-to-value, affordability and a variable rate, plus one new one, a purchase price above our own valuation of the property. LASSO makes the model smaller, not more flexible; if the flexible methods are to earn their place, it will be the next two.

## B2. A decision tree: rules a credit officer can read

Exercise 4 found something the logit cannot say. Mortgages that breach both the affordability rule and the 80 % loan-to-value line have a trouble rate of about 41 %, against 3 to 6 % in the other three cells of the table, so the two ratios do not add, they multiply. A logistic regression with one log-odds term per column can raise the risk along each axis but cannot draw a corner. This is the question the class asked after the linear models, "what about non-linear relationships?"

A decision tree answers it by asking one yes/no question at a time. It looks for the single column and the single threshold that best separate the troubled mortgages from the fine ones, splits the training set in two, and then asks the next question separately in each half. In the class this was the Titanic tree, where two questions, sex and age, were enough to isolate the passengers who did not survive. Two thresholds on two different columns are exactly a corner. Each leaf of the tree holds a group of mortgages and its share of trouble, which is the tree's predicted probability for every application that lands there, and the path from the root to a leaf is a rule that a credit officer can read out and apply by hand.

Two settings control how many questions the tree may ask. `max_depth` limits the number of questions on any path from the root to a leaf, and `min_samples_leaf` refuses any split that would leave a leaf with fewer than that many mortgages. Both exist to stop the tree from memorising the training set, which a tree left to itself will do until every leaf holds a single loan; the third cell below shows what that looks like. A tree compares each value with a threshold and does not care about scale, so it is fitted on the raw dummy columns of Part A2 rather than the standardised ones.

The first cell grows a tree of depth three, draws it, and prints the same tree as text. The second draws the empirical trouble rate over the two ratios as a heat map, which is where the tree looks. The third grows trees of depth 1 to 12 and puts the training AUC next to the test AUC for each.

In [ ]:
# A shallow tree: three questions deep, at least 50 mortgages per leaf
tree3 = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=0).fit(X_train, y_train)
p_tree = tree3.predict_proba(X_test)[:, 1]
print(f"test AUC  tree (depth 3): {roc_auc_score(y_test, p_tree):.3f}   logit: {auc_logit:.3f}")

fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(tree3, feature_names=list(X.columns), class_names=["fine", "trouble"], filled=True,
          impurity=False, proportion=True, rounded=True, fontsize=10, ax=ax)
plt.show()
# The same tree as text: each line is one question, each leaf shows the class it predicts
print(export_text(tree3, feature_names=list(X.columns), show_weights=True, decimals=3))
results["tree (depth 3)"] = (tree3, X_train, X_test)

In [ ]:
# Where the tree looks: the empirical trouble rate over LTV and affordability, with the two rule lines drawn in
grid_ltv = pd.cut(book.ltv, bins=[0.3, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.91])
grid_aff = pd.cut(book.affordability, bins=[0, 0.2, 0.25, 0.3, 1/3, 0.4, 0.5, 2.0])
heat = book.pivot_table(index=grid_aff, columns=grid_ltv, values="trouble_36m", aggfunc="mean", observed=False)
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.heatmap(heat * 100, annot=True, fmt=".0f", cmap="Reds", cbar_kws={"label": "trouble rate (%)"}, ax=ax)
ax.axvline(5, color="black", lw=1.5); ax.axhline(4, color="black", lw=1.5)   # the 80 % LTV line and the one-third line
ax.invert_yaxis()
ax.set_xlabel("loan-to-value"); ax.set_ylabel("affordability ratio")
ax.set_title("Trouble rate (%) by LTV and affordability: the corner the logit cannot draw")
plt.show()

In [ ]:
# Why we limit the depth: deeper trees memorise the training set
depths = range(1, 13)
auc_tr, auc_te = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, min_samples_leaf=5, random_state=0).fit(X_train, y_train)
    auc_tr.append(roc_auc_score(y_train, t.predict_proba(X_train)[:, 1]))
    auc_te.append(roc_auc_score(y_test, t.predict_proba(X_test)[:, 1]))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(depths, auc_tr, marker="o", label="training AUC"); ax.plot(depths, auc_te, marker="o", label="test AUC")
ax.set_xlabel("max_depth"); ax.set_ylabel("AUC"); ax.set_title("Overfitting: the gap opens as the tree deepens"); ax.legend()
plt.show()
print(pd.DataFrame({"depth": depths, "train AUC": np.round(auc_tr, 3), "test AUC": np.round(auc_te, 3)}).to_string(index=False))

In this run the tree's first question is the Swiss loan-to-value rule: is `ltv` at most 0.801? The threshold is the tree's own choice, and it landed on the bank's 80 % line. Of the 7,000 training mortgages, 5,930 answer yes and 3.8 % of them ran into trouble; 1,070 answer no and 24.5 % did. The second question on the high-LTV side is the affordability rule, again with the threshold found by the tree, 0.339, which is the one-third line: 488 mortgages below it with 4.7 % trouble, 582 above it with 41.1 %. That is the corner of Exercise 4, recovered from the data in two questions. The third question inside the corner is `fixed_years <= 2.5`, which in this book means "is it a SARON mortgage", because every fixed-rate mortgage has a term of 5 or 10 years and every SARON mortgage a term of 0. The leaf with the highest trouble share is that one: LTV above 80 %, affordability above one third, variable rate, 203 training mortgages of which 119, or 58.6 %, ran into trouble, and it is the only leaf the tree labels `trouble`. Its fixed-rate neighbour holds 379 mortgages with 31.7 %. On the low-LTV side the tree finds a second, smaller pocket, SARON mortgages whose actual interest burden exceeds 0.095 of income, 69 mortgages with 46.4 % trouble, and among fixed-rate borrowers it separates those over 62 (112 mortgages, 13.4 %) from everyone else, the 3,768 mortgages with 1.6 % trouble where most of the book lives. Three questions, eight leaves, and a test AUC of 0.824 against the logit's 0.816.

The heat map is the same corner without a model. Below 80 % LTV the cells read between 1 and 9 % up to an affordability of 0.5 and 9 to 14 % above it, apart from the left-most column, where the book holds very few loans and a handful of cases move the percentage. Above 80 % LTV but within the affordability rule the cells read 2 to 9 %. Above 80 % LTV and above one-third affordability, the six cells read 33 to 50 %. A logistic regression can raise the risk along each axis; it cannot make those six cells different in kind from their neighbours. Two thresholds do.

The depth sweep shows the price of asking more questions. Training and test AUC rise together up to depth 4, where the test AUC peaks at 0.854 with a training AUC of 0.853. From depth 5 the two part: the training AUC keeps climbing, to 0.971 at depth 12, while the test AUC falls to 0.849, then 0.844, is back at the logit's level by depth 7 (0.816) and ends at 0.647. A tree of depth 12 with at least five loans per leaf has memorised the training set and has lost most of its edge on new loans. The depth-3 tree gives up three points of AUC against the best depth in exchange for eight leaves that fit on one page.

What the analyst tells the CRO: a decision tree of three questions, fitted on the same columns as the logistic regression, rediscovers the bank's own two rules, 80 % loan-to-value and one-third affordability, as its first two questions, adds the rate type as the third, and ranks unseen applications slightly better than the logistic regression (an AUC of 0.824 against 0.816). Its eight leaves are rules a credit officer can apply by hand, and the one that matters says that a variable-rate mortgage that breaches both rules went wrong in 59 of 100 cases in our book. It is the first model in this notebook the officers could run without a computer. Deeper trees memorise the book and do worse on new loans, so if the bank wants a tree, it wants a shallow one.

## B3. A random forest: a panel instead of one interviewer

The class introduced the forest with a hiring analogy. A single tree is one interviewer: it asks its questions in a fixed order, and a slightly different set of candidates would have led it to open with a different question and to reach a different verdict. That instability is where the overfitting in the depth plot comes from. A random forest replaces the interviewer with a panel. Each of several hundred trees is grown on a bootstrap sample of the training set, drawn with replacement, so that every tree sees some mortgages twice and others not at all, and at each question the tree may only choose among a random subset of the qualifications, so that not every tree opens with loan-to-value, as the single tree in B2 did. Every tree is grown deep. The forest's prediction for an application is the verdict of the panel. In the slide's picture each tree votes trouble or fine and the majority wins; what scikit-learn computes is one step finer: each tree reports the trouble share of the leaf the application lands in, and the forest averages those shares, which is the same as the vote share when every leaf is pure and a smoother number when, as here, leaves hold at least five loans. Each tree memorises its own sample, but the trees memorise different things, and the average washes the memorising away. That is why a forest generalises where one deep tree does not.

The price is readability. Nobody can read three hundred trees, and there is no table of coefficients. What we can ask the forest is which columns it *used*. Permutation importance shuffles one column of the test set, so that this column becomes noise while everything else stays as it was, and records how far the AUC drops; the shuffle is repeated a few times and the drops averaged. A column the forest relies on costs points of AUC when it is shuffled; a column the forest ignores costs nothing.

The first cell grows forests of 1, 5, 25, 100 and 300 trees, records the test AUC of each, and keeps the panel of 300. The second asks that panel which columns matter.

In [ ]:
# How many interviewers does the panel need? Test AUC against the number of trees
n_grid = [1, 5, 25, 100, 300]
auc_n = []
for n in n_grid:
    f = RandomForestClassifier(n_estimators=n, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(X_train, y_train)
    auc_n.append(roc_auc_score(y_test, f.predict_proba(X_test)[:, 1]))
print(pd.DataFrame({"trees": n_grid, "test AUC": np.round(auc_n, 3)}).to_string(index=False))

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(X_train, y_train)
p_rf = rf.predict_proba(X_test)[:, 1]
print(f"\ntraining AUC of the forest: {roc_auc_score(y_train, rf.predict_proba(X_train)[:, 1]):.3f}   test AUC: {roc_auc_score(y_test, p_rf):.3f}")
results["random forest"] = (rf, X_train, X_test)

In [ ]:
# Which columns does the forest rely on? Permutation importance on the test set (drop in AUC when a column is shuffled)
imp = permutation_importance(rf, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=0, n_jobs=-1)
imp_s = pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 5))
imp_s.head(10)[::-1].plot.barh(ax=ax)
ax.set_xlabel("drop in test AUC when the column is shuffled"); ax.set_title("Random forest: the ten columns that matter most")
plt.show()
print(imp_s.head(10).round(4).to_string())

In this run one interviewer is a poor judge and a panel is a good one. A single deep tree scores a test AUC of 0.706, five trees 0.807, twenty-five 0.850, and from there the panel gains little: 0.857 with 100 trees and 0.858 with 300. The jump from one tree to twenty-five is the vote at work. Each tree is grown until its leaves hold at least five loans and memorises its own bootstrap sample, and the forest's training AUC of 0.989 shows that the memorising is still there. What the vote removes is its effect on new loans, where the same forest scores 0.858, above the logit (0.816), the LASSO (0.823), the depth-3 tree (0.824) and the best single tree of the depth sweep (0.854 at depth 4). A model that fits the training set almost perfectly and still generalises is something the first two classes had no example of, and it is what the committee has read about.

The permutation importances say that the forest relies on the columns the tree and the logit found. Shuffling `ltv` costs 0.083 of test AUC, shuffling `affordability` 0.046 and shuffling `age` 0.031; `employment_self_employed` follows with 0.027, then `fixed_years` (0.011) and `rate_type_saron` (0.009). Those last two carry the same information, since a SARON mortgage is one with `fixed_years` of 0, so the forest spreads the rate-type signal over both columns and each looks smaller than the pair; shuffling one leaves the other to answer. Below the top six the drops are 0.007 for `actual_burden` and at most 0.002 for the rest, `rooms` and `household_income` among them. Age is the one column that ranks higher here than in the linear models, where it carried a small coefficient (an odds ratio of 0.92 per standard deviation in Part A2). The tree showed why: the risk in age is a step at about 62 rather than a slope, which a linear term cannot express and a forest of trees can.

What the analyst tells the CRO: a random forest of 300 trees ranks unseen applications with an AUC of 0.858, four points above the logistic regression on the same columns and the same test loans, and it does so by relying on the same signals the readable models found, loan-to-value, affordability, age, self-employment and the rate type; there is no hidden column in it. What it lacks is a rule. Nobody can read 300 trees, so a declined customer would be shown the importance chart rather than the question that declined them. Whether four points of AUC are worth that is a question in francs, and the comparison table at the end of Part B puts a number on it.

## B4. Gradient boosting: fix the mistakes step by step

The forest grew its trees independently and let them vote. Boosting grows them one after another, and each tree is given one job: correct the mistakes of everything grown before it. The recipe from the class has five steps. Start with a simple model, here the base rate, which predicts the same 7 % chance of trouble for every mortgage. Find where it is wrong: for every training mortgage, the residual is the difference between what happened, 0 or 1, and what the current model predicts, so the residuals are large and positive for the troubled loans the model underrates and negative for the fine loans it overrates. Train a small tree on those residuals, a stump of one question or a tree of depth three, so that it learns where the current model is too low and where it is too high. Add that tree to the model with a small weight, the learning rate, so that each round corrects only part of the mistake. Repeat with the residuals of the updated model, and combine: the final prediction is the base rate plus every correction, in order (the adding happens on the log-odds scale, as in the logit, not by adding probabilities). Where the forest averages three hundred opinions formed independently, the booster is one opinion revised round after round, and because each tree needs the residuals of the last, the trees cannot be grown in parallel.

Two settings decide how far the correcting goes. The learning rate says how large a step each round takes, and the number of rounds says how many steps there are. A small learning rate needs many rounds to get anywhere; too many rounds, and the trees start correcting noise in the residuals, which is how boosting overfits. Rather than guess the number of rounds, we let the data stop the training: early stopping sets aside a slice of the training set as a validation set, scores the model on it after every round, and stops adding trees once the score has not improved for a given number of rounds. XGBoost, the library the class named, is a fast, regularised implementation of the same idea. It adds a penalty on the size of each tree, draws a random subset of the rows and of the columns for every tree as the forest does, and implements early stopping against a validation set that we hand it explicitly.

The first cell shows the idea in one dimension: trouble against affordability alone, corrected by one stump per round, after 1, 5 and 60 rounds, over the empirical trouble rate in bins of the training set. The second cell fits the real thing on all columns, scikit-learn's histogram gradient boosting with early stopping, and refits it with the number of rounds early stopping found on all 7,000 training rows, so that the model Part B6 cross-validates needs no validation slice of its own. The third does the same with XGBoost. The fourth shows the trade-off between the learning rate and the number of rounds.

In [ ]:
# The idea in one dimension: trouble against affordability. Stage 0 is the base rate; each stage adds one stump.
toy = GradientBoostingClassifier(n_estimators=60, learning_rate=0.1, max_depth=1, random_state=0)
toy.fit(X_train[["affordability"]], y_train)
xs = pd.DataFrame({"affordability": np.linspace(0.08, 0.7, 300)})
stages = {1: None, 5: None, 60: None}
for k, probs in enumerate(toy.staged_predict_proba(xs), start=1):
    if k in stages:
        stages[k] = probs[:, 1]
bins = pd.cut(X_train.affordability, bins=np.arange(0.05, 0.75, 0.05))
emp = y_train.groupby(bins, observed=False).mean()
centers = np.array([b.mid for b in emp.index])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, (k, pr) in zip(axes, stages.items()):
    ax.scatter(centers, emp.values, color="black", s=25, label="empirical trouble rate (train, binned)")
    ax.axhline(y_train.mean(), color="grey", ls=":", label="stage 0: the base rate")
    ax.plot(xs.affordability, pr, color="C3", lw=2, label=f"after {k} stump{'s' if k > 1 else ''}")
    ax.set_xlabel("affordability ratio"); ax.set_title(f"boosting after {k} round{'s' if k > 1 else ''}"); ax.legend(fontsize=8)
axes[0].set_ylabel("probability of trouble")
plt.show()

In [ ]:
# The real thing: histogram gradient boosting with early stopping on a 20% validation slice of the training data
gbm_es = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=1000, max_depth=3, early_stopping=True,
                                        validation_fraction=0.2, n_iter_no_change=30, random_state=0).fit(X_train, y_train)
print(f"early stopping used {gbm_es.n_iter_} rounds")
# Refit with that number of rounds on all training rows (so Part B6 can cross-validate it without a validation slice)
gbm = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=gbm_es.n_iter_, max_depth=3, early_stopping=False,
                                     random_state=0).fit(X_train, y_train)
p_gbm = gbm.predict_proba(X_test)[:, 1]
print(f"test AUC  gradient boosting: {roc_auc_score(y_test, p_gbm):.3f}   random forest: {roc_auc_score(y_test, results['random forest'][0].predict_proba(X_test)[:, 1]):.3f}   logit: {auc_logit:.3f}")
results["gradient boosting"] = (gbm, X_train, X_test)

In [ ]:
# XGBoost: the same idea, the library the slides name. Early stopping needs an explicit validation set.
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0, stratify=y_train)
xgb_es = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8,
                           eval_metric="auc", early_stopping_rounds=30, random_state=0, n_jobs=-1)
xgb_es.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
best_rounds = int(xgb_es.best_iteration) + 1
xgb_model = xgb.XGBClassifier(n_estimators=best_rounds, learning_rate=0.05, max_depth=3, subsample=0.8,
                              colsample_bytree=0.8, random_state=0, n_jobs=-1).fit(X_train, y_train)
p_xgb = xgb_model.predict_proba(X_test)[:, 1]
print(f"XGBoost stopped after {best_rounds} rounds; test AUC {roc_auc_score(y_test, p_xgb):.3f}")
results["XGBoost"] = (xgb_model, X_train, X_test)

In [ ]:
# Learning rate against the number of rounds: small steps need more of them
rows = []
for lr in [0.03, 0.1, 0.3]:
    m = xgb.XGBClassifier(n_estimators=1000, learning_rate=lr, max_depth=3, subsample=0.8, colsample_bytree=0.8,
                          eval_metric="auc", early_stopping_rounds=30, random_state=0, n_jobs=-1)
    m.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
    rows.append({"learning rate": lr, "rounds used": int(m.best_iteration) + 1,
                 "test AUC": round(roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]), 3)})
pd.DataFrame(rows)

In this run the three panels show the recipe at work. After one round the model is the base rate plus one stump: a single step, up to the right of a threshold a little above the one-third line, so that mortgages with a high affordability ratio get a slightly higher probability than the base rate and everyone else a slightly lower one. After five rounds the curve is a staircase of a few steps that rises from below the base rate on the left to roughly twice the base rate on the right, and it still sits well under the black dots at high affordability: five steps with a learning rate of 0.1 correct only a part of the mistake, by design.

After sixty rounds the red curve runs through the dots. It is flat and below the base rate up to an affordability of about 0.3, steep between 0.3 and 0.45, and still rising at the top of the range, where the empirical trouble rate is several times the base rate. Every one of the sixty stumps asked one question about affordability, the curve is the sum of their answers, and it bends where the data bend. The rise starts at the one-third line, which is the Swiss rule again, and a bend of that kind is what a logit's single slope in the log-odds cannot draw.

On all columns, early stopping halted scikit-learn's booster after 150 rounds. That count includes the 30 rounds the rule waits for an improvement on the validation slice before it gives up, so the model had stopped improving around round 120, and the refit on all 7,000 training mortgages keeps the 150. Its test AUC is 0.884, against 0.858 for the random forest and 0.816 for the logit: about two and a half points above the forest and almost seven above the logit, on the same columns and the same test loans. XGBoost, given a validation slice of the same size explicitly and a random 80 % of the rows and of the columns for every tree, stopped after 93 rounds and reaches a test AUC of 0.883, the same number for practical purposes.

The two implementations differ in their bookkeeping (XGBoost reports the best round, scikit-learn the round at which it gave up) and in their regularisation, and they agree on the answer.

The learning-rate table shows the trade-off, with the noise that comes with it. From a learning rate of 0.05 upward, larger steps need fewer rounds: 93 rounds at 0.05, 33 at 0.1 and 13 at 0.3. At 0.03 the rule stopped after 50 rounds, earlier than at 0.05, which runs against the pattern and is the early-stopping rule itself at work: it gives up after 30 rounds without an improvement of the AUC on a validation slice of 1,400 mortgages, of which about a hundred are troubled at the book's 7 % rate, and with steps that small a plateau of 30 rounds is easy to hit before the model is done. The test AUCs are 0.873 at 0.03, 0.879 at 0.1 and 0.871 at 0.3, against 0.883 at 0.05: all within about a point of each other, and all above the forest. The lesson is not that 0.05 is the right learning rate. It is that with early stopping in place the learning rate matters little, and that a count of rounds must always be read together with the rate that produced it.

What the analyst tells the CRO: gradient boosting, in two implementations, ranks unseen applications with an AUC of 0.88, seven points above the logistic regression and two and a half above the random forest, on the same columns and the same test loans, and it needs little hand-tuning beyond a validation slice that tells it when to stop. It is the best ranking in this notebook and, like the forest, it is a model nobody can read; whether the committee should want it is the question the comparison table at the end of Part B answers.

## B5. Support vector machine: the widest street between the two groups

The last method of the class draws no tree and estimates no odds ratio. It looks for the boundary between the two groups, and it chooses the boundary that keeps the greatest distance from both. With one feature, say affordability alone, the boundary is a point on the line, and the margin is the distance from that point to the nearest applicant on either side. With two features it is a line, and the margin is a street along it whose edges touch the nearest applicants of each group; those applicants are the support vectors, and they alone decide where the street runs. Among all the streets that separate the two groups, the support vector machine picks the widest. Real applicants do not separate cleanly, so the street has to allow some of them on the wrong side, and the setting `C` says how much a misclassified applicant may push the street. A large `C` makes every mistake expensive and yields a hard margin, a narrow street bent to accommodate single points; a small `C` tolerates mistakes and yields a soft margin, a wide street that ignores the odd applicant on the wrong side. A straight street cannot carve the corner of Exercise 4 any more than the logit could, and here the kernel enters: the radial basis function, `rbf`, lets the street bend, by measuring the similarity of two applicants instead of their raw coordinates, so that the boundary can close around a region instead of running across the whole plane.

Two practical consequences follow from the geometry. Because the street is defined by distances, every column must be on the same scale, so the SVM is fitted on the standardised columns of Part A2, as the logit was. And what the SVM gives an application is its signed distance to the boundary, not a probability of trouble. For the AUC that is enough, since the AUC needs only the ranking. For the cost calculation of Part B6 we need probabilities that mean what they say, and scikit-learn adds them on request (`probability=True`): it cross-validates the distances within the training set and fits a logistic curve that maps distance to probability, a calibration step that costs a few extra fits.

The first cell keeps two features, loan-to-value and affordability, so that the street can be drawn, and fits four SVMs on a 2,000-mortgage sample of the training set: a linear street, then a bending one at three values of `C`. The classes are weighted equally in these plots (`class_weight="balanced"`), because with 7 % trouble an unweighted street would sit far off to one side and the picture would show nothing. The second cell fits the SVM on all columns of the training set, with the same weighting; the calibration step then maps the weighted distances back to probabilities that mean what they say. It reads the AUC twice, from the distance and from the calibrated probabilities.

In [ ]:
# Two features, so we can see the boundary: LTV and affordability, standardised, on a 2,000-row sample of the training set
two = ["ltv", "affordability"]
sample = X_train_s.sample(2000, random_state=0)
ys = y_train.loc[sample.index]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.8))
configs = [("linear", 1.0), ("rbf", 0.1), ("rbf", 1.0), ("rbf", 10.0)]
for ax, (kernel, C) in zip(axes, configs):
    m = SVC(kernel=kernel, C=C, gamma="scale", class_weight="balanced").fit(sample[two], ys)
    DecisionBoundaryDisplay.from_estimator(m, sample[two], response_method="decision_function", plot_method="contourf",
                                           levels=[-100, 0, 100], alpha=0.2, cmap="coolwarm", ax=ax)
    ax.scatter(sample.ltv[ys == 0], sample.affordability[ys == 0], s=6, alpha=0.3, color="C0", label="fine")
    ax.scatter(sample.ltv[ys == 1], sample.affordability[ys == 1], s=10, alpha=0.7, color="C3", label="trouble")
    ax.set_xlabel("LTV (standardised)"); ax.set_ylabel("affordability (standardised)")
    ax.set_title(f"kernel={kernel}, C={C}"); ax.legend(fontsize=8, loc="lower right")
plt.suptitle("SVM decision boundaries (class_weight='balanced' so the minority class is visible)")
plt.show()

In [ ]:
# The SVM on all features. decision_function gives the signed distance to the boundary; AUC only needs the ranking.
# probability=True is deprecated from scikit-learn 1.9 and removed in 1.11 (CalibratedClassifierCV(SVC(), ensemble=False) then); silence only that
warnings.filterwarnings("ignore", message="The `probability` parameter was deprecated")
svm = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", probability=True, random_state=0).fit(X_train_s, y_train)
p_svm = svm.predict_proba(X_test_s)[:, 1]
print(f"test AUC from the distance to the boundary: {roc_auc_score(y_test, svm.decision_function(X_test_s)):.3f}")
print(f"test AUC from the calibrated probabilities : {roc_auc_score(y_test, p_svm):.3f}")
results["SVM (RBF)"] = (svm, X_train_s, X_test_s)

In this run the four pictures tell the story of the street. The linear street is a straight line running diagonally across the plane, and every applicant to its upper right, with a high loan-to-value ratio or a high affordability ratio or both, is on the trouble side. It can tilt, but it cannot carve the corner of Exercise 4, so to reach the troubled applicants in the corner it must also take in a band of fine applicants with a high loan-to-value ratio and a comfortable affordability ratio. The bending street at C = 0.1 does what the tree did with two questions: one smooth boundary that puts the upper right of the plane, where both ratios are high, on the trouble side and leaves the rest on the fine side. At C = 1 the street starts to bend around individual applicants, and islands of trouble appear in regions that hold a handful of red points; at C = 10 the islands multiply and the boundary is jagged. A hard margin has drawn the sample rather than the book, which is the SVM's version of the deep tree in B2. The picture to keep is the second one; the price of the third and the fourth is the same overfitting as everywhere else in this notebook, in a new costume.

On all columns the SVM lands where the logit is. The AUC from the distance to the boundary is 0.817 and from the calibrated probabilities 0.817, the same ranking, and equal to the logit's 0.816 for practical purposes; it is four points below the forest's 0.858 and almost seven below the boosters' 0.884. The bending street that carved the corner on two columns did not carry that advantage to 29. One candidate reason for the gap to the trees is the kernel, which measures the similarity of two applicants across all 29 standardised columns at once, so that the few columns that carry most of the signal share every distance with many that carry little; another is that the street is fitted with the default `C` of 1 and the default `gamma`, and a grid search over both might close some of the gap. We did not run one, and the other methods of Part B were not tuned beyond the settings shown either. The cell is also a slow one for what it delivers: `probability=True` costs six fits instead of one, five of them for the calibration, and a kernel SVM's fitting time grows roughly with the square of the number of rows, so on a book ten times this size it would be the method the bank waits for.

What the analyst tells the CRO: a support vector machine with a bending boundary can draw the corner between loan-to-value and affordability as well as the tree can, and the pictures show it; on all columns, with the classes weighted and without further tuning, it ranks unseen applications as well as the logistic regression (an AUC of 0.817 against 0.816) and no better, it gives a probability only through an extra calibration step, it explains nothing, and it is slow to fit. On this book it offers what the logit offers, without the odds ratios. We keep it in the table so that the committee sees that "machine learning" is not one thing.

## B6. The comparison the CRO asked for

Seven models are now fitted, and the CRO asked for one table. Every row of it is judged on the same 3,000 test mortgages, with the two numbers of Part A2: the AUC of the model's ranking, and the expected cost in CHF per 1,000 applications of its decisions at the model's own approval bar. The bar is the one place where a comparison can quietly go wrong. Exercise 3 chose the logit's threshold by sweeping the test set, which was fine for one model but would not be fair for seven: a threshold tuned on the test set has seen the answers, and a model with a more jagged cost curve would profit more from the peeking. Here each model's threshold is chosen **within the training set**: `cross_val_predict` splits the 7,000 training mortgages into five folds, refits a copy of the model on four of them and predicts the fifth, until every training mortgage has a prediction from a model that did not see it, and `best_threshold` then sweeps those predictions for the cost-minimising bar. The test set is touched once per model, to compute the AUC and the cost at that bar.

The table has six columns. `test AUC` is the ranking quality, threshold-free. `threshold` is the bar chosen on the cross-validated training predictions. `recall` and `precision` describe the decisions at that bar on the test set: the share of troubled loans the model declines, and the share of declined loans that would have been troubled. `cost per 1,000 (CHF)` is the CRO's number. `explainable` is not a metric; it is our judgement of what the bank could show a supervisor or a declined customer: odds ratios, a set of rules, an importance chart, or nothing. The first row is the baseline that approves everyone, so that every model has the number it must beat next to it. The comparison cell refits every model five times, so it takes a few minutes on Colab; it prints the name of each model as it goes.

In [ ]:
# Thresholds from 5-fold cross-validated training predictions, then one evaluation on the test set per model
from sklearn.base import clone
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
explainable = {"logit": "yes (odds ratios)", "LASSO logit": "yes (fewer odds ratios)", "tree (depth 3)": "yes (rules)",
               "random forest": "partly (importances)", "gradient boosting": "partly (importances)",
               "XGBoost": "partly (importances)", "SVM (RBF)": "no"}
thresholds, rows = {}, []
for name, (model, Xtr, Xte) in results.items():
    print(f"cross-validating {name} ...")
    cv_prob = cross_val_predict(clone(model), Xtr, y_train, cv=cv, method="predict_proba")[:, 1]
    t = best_threshold(y_train, cv_prob)
    thresholds[name] = t
    p = model.predict_proba(Xte)[:, 1]
    pred = (p >= t).astype(int)
    rows.append({"model": name, "test AUC": roc_auc_score(y_test, p), "threshold": t,
                 "recall": recall_score(y_test, pred), "precision": precision_score(y_test, pred, zero_division=0),
                 "cost per 1,000 (CHF)": cost_per_1000(y_test, p, t), "explainable": explainable[name]})
majority = {"model": "approve everyone", "test AUC": 0.5, "threshold": np.nan, "recall": 0.0, "precision": np.nan,
            "cost per 1,000 (CHF)": cost_per_1000(y_test, np.zeros(len(y_test)), 0.5), "explainable": "yes"}
comparison = pd.DataFrame([majority] + rows).set_index("model")
comparison.round(3)

In [ ]:
# The same table as a picture: AUC and cost side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
comparison["test AUC"].plot.barh(ax=axes[0]); axes[0].set_xlim(0.45, 0.95); axes[0].set_title("test AUC"); axes[0].axvline(0.5, color="grey", ls=":")
comparison["cost per 1,000 (CHF)"].plot.barh(ax=axes[1]); axes[1].set_title("expected cost per 1,000 applications (CHF)")
plt.tight_layout(); plt.show()

### Reading the table as the CRO

In this run the AUC column has three tiers. The two boosters lead, 0.884 for scikit-learn's gradient boosting and 0.883 for XGBoost; the forest follows at 0.858; and then come the three readable models and the SVM close together, 0.824 for the tree, 0.823 for the LASSO, 0.817 for the SVM and 0.816 for the logit. The cost column tells almost the same story with a different order at the top. Approving everyone costs CHF 4,160,000 per 1,000 applications. The logit at its cross-validated bar of 0.11 brings that down to CHF 2,690,667, the LASSO to CHF 2,634,667, the tree to CHF 2,338,667, the forest to CHF 1,822,667, gradient boosting to CHF 1,837,333 and XGBoost to CHF 1,766,667; the SVM, at CHF 2,732,000, is the one model that costs more than the logit, by CHF 41,333 at the same bar of 0.11 and with the same AUC.

The cheapest model is XGBoost. It saves CHF 924,000 per 1,000 applications against the logit and CHF 2,393,333 against approving everyone, and since the bank has 500 applications on its desk this year, the difference to the logit on this year's intake is about CHF 460,000.

Three things in the table deserve a second look.

First, the three ensembles are within CHF 71,000 of each other, and the highest AUC, gradient boosting's, does not give the lowest cost: the forest, with an AUC two and a half points lower, costs CHF 14,667 less. On 3,000 test loans one approved troubled loan moves the cost by CHF 20,000 per 1,000 applications and one declined good loan by CHF 2,667, so the gaps among the three ensembles amount to a few loans and mean nothing, while the gap between the three and the logit, worth between 43 and 46 troubled loans, does. The two columns also measure different things: the AUC is threshold-free, while the cost is one operating point, chosen on the training folds, so a model with the better ranking can lose by a few loans at its own bar.

Second, the thresholds are honest and therefore not the one Exercise 3 found. Chosen on cross-validated training predictions, the logit's bar is 0.11, close to the break-even probability of 12 % from Part A2, and the cost at that bar is what the bank would have got had it set the bar before seeing the test loans, which is a little worse than the bar Exercise 3 found by sweeping them. Each model gets its own bar, 0.13 for the boosters, 0.15 for the forest and 0.17 for the tree, because each model's probabilities are calibrated differently, and a bar that is right for one is wrong for another.

Third, the readable tree does more than a third of the work. It cuts the cost by CHF 352,000 against the logit, out of the CHF 924,000 that XGBoost gains, with eight rules on a page. The recall and precision columns show how: at its bar of 0.17 it declines fewer applicants than XGBoost and catches fewer of the troubled loans (recall 0.538 against 0.736), but it is right about the ones it declines more often (precision 0.416 against 0.380).

### Why the flexible models win here

The gain is not magic, and Part B has shown where it comes from. Exercise 4 found that the two Swiss rules multiply rather than add: mortgages that breach both the one-third affordability line and the 80 % loan-to-value line ran into trouble at about 41 %, against 3 to 6 % in the other three cells. A logistic regression adds one log-odds term per column, and no sum of straight lines makes a corner. The LASSO, which is the same straight line with fewer terms, gained less than a point of AUC, and confirms that the problem was never the number of columns. The tree drew the corner with two questions and gained the CHF 352,000. The forest and the boosters found the rest: the rate type inside the corner, where a variable rate turned a 32 % pocket into a 59 % one in the tree; the step in age at about 62 that the tree found and that the forest's importances then flagged as its third column, a step the logit's slope could not express; the affordability effect, which the B4 panels showed to be flat up to the one-third line and steep beyond it, a bend and not a slope; and the combinations of all of these that eight leaves cannot hold and three hundred trees can. Every one of these is a shape a straight line cannot draw, and all of them are made of the same columns the logit had. The flexible methods did not find new information in the book. They found the shape of the information that was there.

### Would we deploy it?

The CRO asked two questions, and the table answers only the first. The second is whether the bank could defend the model, to its supervisor and to a customer whose application it declined. For the logit and the LASSO the answer is the table of odds ratios: the supervisor can audit every coefficient, and the declined customer can be told that the affordability ratio and the loan-to-value ratio put the application above the bar. For the tree it is one rule read off the page, which is the best answer of all. For the forest and the boosters it is an importance chart, which says which columns the model relies on but not what it did with them for this application; the customer cannot be told the question that declined them, because there is no such question. The SVM offers nothing. That is the `explainable` column, and it is a judgement, not a metric.

A model nobody can read is a model risk, and the supervisor will say so. If the bank deploys the booster, three things come with it. It needs monitoring, because a model that learned the book of 2019 to 2022 will drift as rates, prices and borrowers change, and nobody will see the drift by looking at the model. It needs a validation set that is refreshed as this year's outcomes come in, so that the AUC and the cost in the table are recomputed every year on loans the model has never seen. And it needs the logit next to it as a champion-challenger pair: the readable model scores every application in parallel, the two rankings are compared, and the cases where they disagree are the ones a credit officer looks at by hand. None of that is free, and the CHF 924,000 per 1,000 applications is what it has to pay for.

The honest position is this. On this book, on unseen loans, the best flexible model is worth about CHF 924,000 per 1,000 applications more than the logistic regression, and a readable tree of eight rules captures CHF 352,000 of that on its own. The flexible models earn their gain from the shape of information the bank already had, and the gain comes with a model the bank cannot read out at the counter. Whether the difference is worth the model risk is not a data question, and the choice is the CRO's.

What the analyst tells the CRO: on 3,000 unseen loans the boosters rank applications best (an AUC of 0.88 against 0.82 for the logistic regression) and the cheaper of the two, XGBoost, costs the bank about CHF 924,000 per 1,000 applications less at an approval bar set on the training data alone; more than a third of that gain is available from a decision tree of eight rules that a credit officer can read. Our recommendation is to put the tree's rules into the credit policy now, run the booster beside the logistic regression as a challenger for a year with a refreshed validation set, and decide on deployment when we can show the committee that its gain holds on loans that did not exist when it was trained. Part C, Exercises 5 to 8, hands the same tools to your team.

# Part C: Exercises on the new methods

The CRO has read the table. What the table does not yet contain is a decision, and the four exercises of this part turn it into one: whether the untuned settings of Part B were a fair trial of the flexible methods, what the models say about the 500 applications on the bank's desk this year and where they disagree, what happens to the book if variable rates rise, and the memo that puts the answer on one page. Each exercise uses the objects the worked cells created, `logit`, `gbm`, `tree3`, `thresholds` and the training and test sets of Part A2, so run the notebook up to here before you start, and keep the given cells as they are.

One given cell comes first. The applications file has the same columns as the book, but a model does not see columns, it sees the matrix it was trained on: the same dummy columns, in the same order, with a canton that happens to have no application this year still present as a column of zeros, and, for the logit and the SVM, the same standardisation, with the mean and the standard deviation of the *training* set, not of the applications. The helper below encodes the applications exactly like the book, with the dummy columns of `X` and the `scaler` of Part A2. Anything that arrives at the counter must go through it before a model scores it, and the stress test of Exercise 7 will need it a second time.

In [ ]:
# Given: encode the applications like the book. Missing dummy columns (a canton with no application) are filled with 0.
def encode_like_book(df):
    Xa = pd.get_dummies(df[features], drop_first=True).astype(float)
    return Xa.reindex(columns=X.columns, fill_value=0.0)

X_apps = encode_like_book(apps)
X_apps_s = pd.DataFrame(scaler.transform(X_apps), columns=X.columns, index=X_apps.index)
print("applications encoded:", X_apps.shape, "| same columns as the book:", list(X_apps.columns) == list(X.columns))

### Exercise 5: Tune the ensembles properly

Use `GridSearchCV` with 5-fold stratified cross-validation (`scoring="roc_auc"`) on the **training set** to tune (a) the random forest over `max_depth` in {4, 8, None} and `min_samples_leaf` in {1, 5, 20} (with `n_estimators=200`), and (b) `HistGradientBoostingClassifier` over `learning_rate` in {0.03, 0.1, 0.3} and `max_depth` in {2, 3, 5} (with `max_iter=200`, `early_stopping=False`). Show the cross-validated AUC for every setting of each grid. Then report the **test** AUC of the two winners once.

*Deliverable:* two small tables (setting, CV AUC), the two test AUCs, and one sentence on whether tuning changed the ranking in the comparison table.

In [ ]:
# Exercise 5: your code here
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

### Exercise 6: Score this year's applications

Score `applications_2026.csv` with the logit (on `X_apps_s`) and with the gradient booster `gbm` (on `X_apps`), each at its own threshold from `thresholds`. Count how many applications each model would reject and how many the two models disagree on. Pick three applications the booster rejects and the logit approves, show their `ltv`, `affordability`, `rate_type`, `employment`, `age` and `actual_burden`, and explain each disagreement with the depth-3 tree's rules from Part B2 (`tree3.apply(...)` tells you which leaf an application lands in).

*Deliverable:* the counts, the three rows, and one sentence per row on why the flexible model says no.

In [ ]:
# Exercise 6: your code here
# p_apps_logit = logit.predict_proba(X_apps_s)[:, 1]

### Exercise 7: Stress test: SARON rates rise by two percentage points

Copy the applications, add 2.0 percentage points to `interest_rate` for the SARON loans only, recompute `actual_burden = interest_rate / 100 * loan_amount / household_income`, and re-encode. Report the mean predicted probability of trouble and the number of rejections before and after the shock, under the logit and under the booster.

*Deliverable:* a 2×2 table (model × before/after) of the mean predicted trouble rate, and one sentence on which model reacts more strongly and why that is plausible (think about which borrowers a rate rise actually hits).

In [ ]:
# Exercise 7: your code here
# stressed = apps.copy()

### Exercise 8: Five bullets for the CRO

Write the memo. Five bullets, in the cell below (a markdown cell): which model the bank should use for the approval decision and which one beside it as a check, the threshold in words ("we stop approving above a predicted trouble probability of x %"), the expected cost per 1,000 applications against approving everyone, what the bank must monitor after deployment, and one caveat about learning from simulated data.

In [ ]:
# Exercise 8: your code here
# This exercise has no code. Double-click the next cell and write your memo there.

**Memo to the CRO** (your five bullets here)

- 
- 
- 
- 
- 

## Reflection questions (for the class discussion)

1. Mortgages with a high LTV run into trouble more often. Does a high LTV *cause* trouble? What would the bank have to do to find out, and why can none of the models in this notebook answer it (recall the prediction-versus-causality interlude of the first class)?
2. Which column would you look for first as a leakage suspect in the bank's real systems, and how would you test whether a column is known at decision time?
3. The comparison table chose thresholds on cross-validated training predictions and evaluated on the test set once. Exercise 3 chose the logit's threshold on the test set itself. Why does that matter, and in which direction is Exercise 3's cost biased?
4. What would change if the data were real: which of the findings above would you expect to survive, and which numbers would you refuse to quote?
5. A rejected applicant asks why. What can the bank say under each model?

## Recommended reading

- James, Witten, Hastie, Tibshirani, Taylor: *An Introduction to Statistical Learning with Applications in Python*, chapters 3 and 4 (linear and logistic regression), 6 (LASSO), 8 (trees, forests, boosting) and 9 (SVM). Free at statlearning.com.
- FINMA Circular 2017/7 (credit risks, banks) and the SBA self-regulation on mortgage financing, for where the one-third affordability rule and the 80 % LTV standard come from.